In [ ]:
import json
import sys

from pathlib import Path
from datetime import datetime, timezone

import websocket

# ==============================================================================
# Import Protobuf Decoder
# ==============================================================================
PROTO_PATH = Path(
    r"D:\Data Projects\MEXC API Architecture\websocket-proto"
)

sys.path.append(str(PROTO_PATH))

from PushDataV3ApiWrapper_pb2 import PushDataV3ApiWrapper

# ==============================================================================
# WebSocket Configuration
# ==============================================================================
WS_URL = "wss://wbs-api.mexc.com/ws"

SYMBOL = "ETHUSDT"

INTERVAL = "Min1"

# Correct format:
# spot@public.kline.v3.api.pb@{SYMBOL}@{INTERVAL}
ENDPOINT = (
    f"spot@public.kline.v3.api.pb@{SYMBOL}@{INTERVAL}"
)

subscription = {
    "method": "SUBSCRIPTION",
    "params": [
        ENDPOINT
    ],
    "id": 1
}

received_time = lambda: datetime.now(
    timezone.utc
).strftime(
    "%Y-%m-%d %H:%M:%S.%f UTC"
)

# ==============================================================================
# Connect
# ==============================================================================
ws = websocket.create_connection(
    WS_URL
)

ws.send(
    json.dumps(subscription)
)

print(
    "Subscribed:",
    ENDPOINT
)

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets
import time

# ==============================================================================
# Dashboard Output Area
# ==============================================================================
dashboard = widgets.Output(
    layout={
        "border": "1px solid black",
        "height": "400px",
        "overflow_y": "auto"
    }
)

display(dashboard)

# ==============================================================================
# Receive Loop
# ==============================================================================

last_display_update = 0
DISPLAY_REFRESH_RATE = 1.0

while True:

    message = ws.recv()

    if isinstance(message, str):
        continue

    wrapper = PushDataV3ApiWrapper()

    wrapper.ParseFromString(
        message
    )

    kline = wrapper.publicSpotKline

    current_time = time.time()

    if current_time - last_display_update >= DISPLAY_REFRESH_RATE:

        with dashboard:

            clear_output(wait=True)

            print("=" * 80)
            print("ETHUSDT 1 MINUTE OHLCV")
            print("=" * 80)

            print(
                f"Exchange Time : "
                f"{datetime.fromtimestamp(wrapper.createTime/1000, timezone.utc)}"
            )

            print()
            print(f"Open          : {float(kline.openingPrice):,.2f}")
            print(f"High          : {float(kline.highestPrice):,.2f}")
            print(f"Low           : {float(kline.lowestPrice):,.2f}")
            print(f"Close         : {float(kline.closingPrice):,.2f}")

            print()

            print(
                f"Volume        : "
                f"{float(kline.volume):,.5f} ETH"
            )

            print(
                f"Amount        : "
                f"{float(kline.amount):,.2f} USDT"
            )

            print("=" * 80)

        last_display_update = current_time